# 01. Optimización y Regularización de Redes Neuronales con PyTorch
**Autor:** Oscar David  
**Curso / CADI:** Percepción Computacional / Redes Neuronales  
**Repositorio:** https://github.com/oscar2121/redes-neuronales-3

### Objetivo:
Implementar y comparar una red neuronal base frente a una red regularizada y optimizada en PyTorch, evaluando el impacto de Dropout, L2 (Weight Decay), Batch Normalization, Adam y Early Stopping.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Generación y división 70/15/15
X, y = make_classification(n_samples=3000, n_features=30, n_informative=18, n_redundant=8, n_classes=2, flip_y=0.07, random_state=SEED)
X_tr_raw, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.3, random_state=SEED, stratify=y)
X_dev_raw, X_te_raw, y_dev, y_te = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=SEED, stratify=y_tmp)

scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr_raw)
X_dev = scaler.transform(X_dev_raw)
X_te = scaler.transform(X_te_raw)

train_loader = DataLoader(TensorDataset(torch.tensor(X_tr, dtype=torch.float32), torch.tensor(y_tr, dtype=torch.long)), batch_size=64, shuffle=True)
dev_loader = DataLoader(TensorDataset(torch.tensor(X_dev, dtype=torch.float32), torch.tensor(y_dev, dtype=torch.long)), batch_size=64, shuffle=False)
test_loader = DataLoader(TensorDataset(torch.tensor(X_te, dtype=torch.float32), torch.tensor(y_te, dtype=torch.long)), batch_size=64, shuffle=False)

print(f'Train: {X_tr.shape[0]} | Dev: {X_dev.shape[0]} | Test: {X_te.shape[0]}')

### Definición de Modelos (Base vs Regularizado)

In [ ]:
class ModeloBase(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(30, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 2)
        )
    def forward(self, x): return self.net(x)

class ModeloRegularizado(nn.Module):
    def __init__(self, drop=0.35):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(30, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(drop*0.7),
            nn.Linear(32, 2)
        )
    def forward(self, x): return self.net(x)

m_base = ModeloBase()
m_reg = ModeloRegularizado(0.35)
crit = nn.CrossEntropyLoss()
opt_base = optim.SGD(m_base.parameters(), lr=0.03)
opt_reg = optim.Adam(m_reg.parameters(), lr=0.003, weight_decay=1e-4) # L2
sched = optim.lr_scheduler.CosineAnnealingLR(opt_reg, T_max=75, eta_min=1e-5)

### Entrenamiento y Evaluación

In [ ]:
# Entrenamiento Modelo Regularizado con Early Stopping
best_loss, patience, count = float('inf'), 12, 0
best_state = None

for ep in range(1, 76):
    m_reg.train()
    for bx, by in train_loader:
        opt_reg.zero_grad(); out = m_reg(bx); loss = crit(out, by)
        loss.backward(); opt_reg.step()
    sched.step()

    m_reg.eval()
    d_loss, tot = 0.0, 0
    with torch.no_grad():
        for bx, by in dev_loader:
            d_loss += crit(m_reg(bx), by).item() * bx.size(0); tot += bx.size(0)
    cur_loss = d_loss / tot
    if cur_loss < best_loss:
        best_loss = cur_loss; best_state = m_reg.state_dict().copy(); count = 0
    else:
        count += 1
        if count >= patience:
            print(f'Early Stopping en época {ep} (Loss: {best_loss:.4f})')
            m_reg.load_state_dict(best_state)
            break

# Evaluación final en Test Set
m_reg.eval()
with torch.no_grad():
    preds = m_reg(torch.tensor(X_te, dtype=torch.float32)).argmax(dim=1).numpy()
print(f'Accuracy en Test: {accuracy_score(y_te, preds):.4f}')
print(f'F1-Score en Test: {f1_score(y_te, preds):.4f}')